# Capstone Evaluation - Affordable Child Care Matching (USCIC)
**COM SCI-X 455 - Final Capstone**

Runs the capstone's extraction + matching pipeline on 12 multilingual parent answers (`eval/cases.json`) and
measures **field accuracy, privacy violations, match rate and cost per request** for:
- a rule-based baseline (no LLM),
- Claude Sonnet 4.5 and Claude Haiku 4.5 (needs `ANTHROPIC_API_KEY` in Colab Secrets),
- GPT-4o (needs `UCLA_OPENAI_API_KEY` or `OPENAI_API_KEY`).

CPU runtime is enough.

In [ ]:
!git clone -q https://github.com/MyTeachers123/genai-course.git
%cd genai-course/capstone
!pip install -q -r requirements.txt tabulate

In [ ]:
import os
def get_secret(*names):
    """Read a key from Colab Secrets (or env vars). Keys are never printed."""
    try:
        from google.colab import userdata
        for n in names:
            try:
                v = userdata.get(n)
                if v:
                    return v
            except Exception:
                pass
    except ImportError:
        pass
    for n in names:
        if os.environ.get(n):
            return os.environ[n]
    return None

OPENAI_KEY = get_secret("UCLA_OPENAI_API_KEY", "OPENAI_API_KEY")
ANTHROPIC_KEY = get_secret("ANTHROPIC_API_KEY", "UCLA_ANTHROPIC_API_KEY", "CLAUDE_API_KEY")
if OPENAI_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_KEY
if ANTHROPIC_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_KEY
print("OpenAI key found:", bool(OPENAI_KEY), "| Anthropic key found:", bool(ANTHROPIC_KEY))

In [ ]:
import importlib, json, pandas as pd
import app.extract as ex
from eval import run_eval

CONFIGS = [("rule-based baseline", "mock", None)]
if ANTHROPIC_KEY:
    CONFIGS += [("Claude Sonnet 4.5", "claude", "claude-sonnet-4-5"), ("Claude Haiku 4.5", "claude", "claude-haiku-4-5")]
if OPENAI_KEY:
    CONFIGS += [("GPT-4o", "openai", "gpt-4o")]

summaries, details = [], {}
for label, backend, model in CONFIGS:
    os.environ["CAPSTONE_LLM"] = backend
    if model:
        os.environ["CAPSTONE_MODEL"] = model
    df, s = run_eval.run()
    s["config"] = label
    summaries.append(s); details[label] = df
    print(f"===== {label}"); print(df[["id", "result", "correct_fields", "all_correct", "privacy_ok", "matches", "cost_usd", "detail"]].to_markdown(index=False))
summary = pd.DataFrame(summaries)[["config", "field_accuracy", "records_fully_correct", "privacy_violations", "match_rate", "avg_cost_per_request_usd", "cost_per_1000_requests_usd"]]
summary

In [ ]:
# Per-field accuracy: which fields are hard?
fields = ["language", "zip_code", "monthly_budget_usd", "age_groups", "days", "start", "end"]
per_field = pd.DataFrame({label: [df[f"ok_{f}"].fillna(False).astype(bool).sum() if f"ok_{f}" in df else 0 for f in fields]
                          for label, df in details.items()}, index=fields)
per_field.loc["(cases)"] = [len(df) for df in details.values()]
print(per_field.to_markdown())

with open("eval/results.md", "w") as f:
    f.write("# Capstone evaluation results\n\n## Summary\n\n" + summary.to_markdown(index=False) +
            "\n\n## Correct answers per field (12 cases)\n\n" + per_field.to_markdown() + "\n")
    for label, df in details.items():
        f.write(f"\n## {label}\n\n" + df[["id", "result", "correct_fields", "all_correct", "privacy_ok", "matches", "cost_usd", "detail"]].to_markdown(index=False) + "\n")
print("saved eval/results.md")
try:
    from google.colab import files; files.download("eval/results.md")
except Exception:
    pass